# Table II · Auditing the loop: hackability per attack space and agreement with human ratings

**In the paper ([arXiv:2610.10899](https://arxiv.org/abs/2610.10899)):** Table II and the Sec. V paragraphs "Patching hardens narrowly" and "No transfer, and one real trade".

For each predictor: its reconstruction offset R(x̂) − R(x), the attack success rate (ASR) of a fresh EnCodec
attacker, a fresh DAC attacker and waveform PGD, and its agreement with human ratings (BVCC system-level SRCC and the
mean over the 12 out-of-domain sets). An attack succeeds on an utterance when the score rises by more than τ = 0.3
over the codec reconstruction x̂ while at least one independent measure (ESTOI, SI-SDR, WER, SpeechBERTScore) says
the audio got worse.

Recomputes E1 (24 jobs, ~15 min each; stored attackers, no training) and E2 (24 jobs, ~3 min each).

In [1]:
import os, sys
from pathlib import Path

# Find the repository root from wherever the notebook runs (notebooks/ or a subfolder).
ROOT = Path.cwd().resolve()
while not (ROOT / "src" / "hackpatch").is_dir():
    if ROOT.parent == ROOT:
        raise RuntimeError("start the notebook from inside the repository")
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
from IPython.display import Markdown, display
from hackpatch import repro

pd.set_option("display.width", 200, "display.max_columns", 40, "display.max_rows", 200)

In [2]:
# Parameters (also settable through environment variables for non-interactive runs)
# SOURCE = "recompute": re-run the evaluations behind this output from the released checkpoints (GPU), then aggregate.
# SOURCE = "released" : aggregate the released outputs of the original runs (CPU, seconds). This rewrites the
#                       committed analysis/out/ and paper/ files, which should come out identical.
SOURCE = os.environ.get("NB_SOURCE", "recompute")
# RUN = "slurm": one GPU job per run (sbatch), this kernel waits; "local": run in this kernel (needs a GPU).
RUN = os.environ.get("NB_RUN", "slurm")

In [3]:
reg = repro.registry()
KEYS = [k for k, v in reg.items() if v["kind"] in ['e1', 'e2'] and not k.startswith("e2_abl")]
print(len(KEYS), "runs:", ", ".join(KEYS))

48 runs: e2_R0_s0, e2_R0_s1, e2_R0_s2, e1_R0_s0, e1_R3_s0, e1_R0_s1, e1_R3_s1, e1_R0_s2, e1_R3_s2, e1_baseline_noise_s0, e1_baseline_pgd_s0, e1_baseline_random_s0, e1_baseline_noise_s1, e1_baseline_pgd_s1, e1_baseline_random_s1, e1_baseline_noise_s2, e1_baseline_pgd_s2, e1_baseline_random_s2, e1_R1_s0, e1_R2_s0, e1_R1_s1, e1_R2_s1, e1_R1_s2, e1_R2_s2, e2_R1_s0, e2_R2_s0, e2_R3_s0, e2_baseline_noise_s0, e2_baseline_pgd_s0, e2_baseline_random_s0, e2_R1_s1, e2_R2_s1, e2_R3_s1, e2_baseline_noise_s1, e2_baseline_pgd_s1, e2_baseline_random_s1, e2_R1_s2, e2_R2_s2, e2_R3_s2, e2_baseline_noise_s2, e2_baseline_pgd_s2, e2_baseline_random_s2, e1_R3ms_s0, e2_R3ms_s0, e1_R3ms_s1, e2_R3ms_s1, e1_R3ms_s2, e2_R3ms_s2


In [4]:
if SOURCE == "recompute":
    repro.run_keys(KEYS, "recompute", RUN)  # skips runs whose recomputed output already exists
dirs = repro.run_analysis(SOURCE)  # whole analysis chain (seconds); falls back to released outputs where needed
OUT, PAPER = dirs["out"], dirs["paper"]
COMMITTED_OUT, COMMITTED_PAPER = ROOT / "analysis" / "out", ROOT / "paper"

[analysis] runs=/fs/scratch/PAS2301/alialavi/cache/audio_assessment_hackability_release/runs
[analysis] out=/users/PAS2301/alialavi/projects/audio_assessment_hackability/analysis/out
[analysis] paper=/users/PAS2301/alialavi/projects/audio_assessment_hackability/paper


In [5]:
# Where this notebook's inputs came from: recomputed from checkpoints, or released outputs of the original runs.
used = repro.load_registry(repro.results_registry(SOURCE))
pd.Series({k: used[k].get("source", "released") for k in KEYS}, name="source").value_counts()

source
released    48
Name: count, dtype: int64

In [6]:
print((PAPER / "tables" / "table1.tex").read_text())

% generated by analysis/paper_assets.py
\begin{tabular}{l c c c c c c}
\toprule
& offset & \multicolumn{3}{c}{ASR by attack space} & \multicolumn{2}{c}{human agreement} \\
\cmidrule(lr){3-5}\cmidrule(lr){6-7}
Predictor & $R(\hat{x})\!-\!R(x)$ & EnCodec & DAC & PGD & BVCC & OOD \\
\midrule
$R_0$ (no defense) & -0.09$\pm$0.02 & 0.226$\pm$0.205 & 0.065$\pm$0.113 & 0.999$\pm$0.002 & 0.922$\pm$0.004 & 0.731$\pm$0.006 \\
$R_1$ & -0.06$\pm$0.03 & 0.050$\pm$0.048 & 0.161$\pm$0.033 & 1.000$\pm$0.001 & 0.911$\pm$0.009 & 0.690$\pm$0.014 \\
$R_2$ & 0.00$\pm$0.07 & 0.017$\pm$0.006 & 0.141$\pm$0.025 & 1.000$\pm$0.000 & 0.918$\pm$0.003 & 0.682$\pm$0.008 \\
$R_3$ (EnCodec loop) & -0.27$\pm$0.25 & 0.137$\pm$0.179 & 0.127$\pm$0.031 & 1.000$\pm$0.000 & 0.920$\pm$0.001 & 0.637$\pm$0.046 \\
$R_3^{\mathrm{multi}}$ (multi-space) & 0.95$\pm$0.16 & 0.082$\pm$0.107 & 0.016$\pm$0.026 & 0.841$\pm$0.089 & 0.914$\pm$0.003 & 0.552$\pm$0.015 \\
noise pairs & -0.09$\pm$0.01 & 0.270$\pm$0.107 & 0.230$\pm$0.028 & 1.000$

In [7]:
print(repro.diff_text(PAPER / "tables" / "table1.tex", COMMITTED_PAPER / "tables" / "table1.tex"))

identical to the committed file


In [8]:
# Numbers the paper quotes from this output: committed value vs this notebook's value.
cmp = repro.compare_numbers(OUT / "paper_numbers.json", prefixes=('e1_', 'ood_sys_srcc', 'r0_bvcc'))
print(f"{len(cmp)} numbers; max |diff| = {cmp.abs_diff.max():.4f}; identical: {(cmp.abs_diff == 0).sum()}")
cmp.head(20)

1338 numbers; max |diff| = 0.0000; identical: 1338


,paper,reproduced,abs_diff
e1_R0_grpo_dac_asr_ref_input_2dp,0.120,0.120,0.0
e1_R0_grpo_dac_asr_ref_input_3dp,0.120,0.120,0.0
e1_R0_grpo_dac_asr_ref_input_sd,0.192,0.192,0.0
e1_R0_grpo_dac_asr_ref_recon_2dp,0.070,0.070,0.0
e1_R0_grpo_dac_asr_ref_recon_3dp,0.065,0.065,0.0
e1_R0_grpo_dac_asr_ref_recon_sd,0.113,0.113,0.0
e1_R0_grpo_dac_ceiling_ref_recon_2dp,0.070,0.070,0.0
e1_R0_grpo_dac_ceiling_ref_recon_3dp,0.066,0.066,0.0
e1_R0_grpo_dac_ceiling_ref_recon_sd,0.114,0.114,0.0
e1_R0_grpo_dac_estoi_delta_2dp,-0.010,-0.010,0.0
